# Customer Purchase Behavior Analysis using Markov Chain

## 1. Introduction

This notebook analyzes customer event behavior and models journey transitions using a first-order Markov Chain.

Goal:
- Build clean event sequences per visitor
- Estimate transition probabilities
- Predict the next likely state

In [2]:
## 2. Data Processing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

csv_path = Path("../Data/events.csv")
raw_df = pd.read_csv(csv_path)

# Keep required columns and target events only
df = raw_df[["visitorid", "timestamp", "event"]].copy()
df = df[df["event"].isin(["view", "addtocart", "transaction"])].copy()

# Convert milliseconds epoch to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

# Sort by user journey order
df = df.sort_values(["visitorid", "timestamp"]).reset_index(drop=True)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nEvent distribution:")
print(df["event"].value_counts())

Shape: (2756101, 3)

First 5 rows:
   visitorid               timestamp event
0          0 2015-09-11 20:49:49.439  view
1          0 2015-09-11 20:52:39.591  view
2          0 2015-09-11 20:55:17.175  view
3          1 2015-08-13 17:46:06.444  view
4          2 2015-08-07 17:51:44.567  view

Event distribution:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


## 3. State Mapping

state_map = {
    "view": "Browse",
    "addtocart": "Add_to_Cart",
    "transaction": "Purchase",
}

df["state"] = df["event"].map(state_map)
print(df[["visitorid", "timestamp", "event", "state"]].head())

In [3]:
## 4. Sequence Modeling

# Safety check: recreate state if this cell runs before State Mapping
if "state" not in df.columns:
    state_map = {
        "view": "Browse",
        "addtocart": "Add_to_Cart",
        "transaction": "Purchase",
    }
    df["state"] = df["event"].map(state_map)

sequences = (
    df.groupby("visitorid")["state"]
      .apply(list)
      .apply(lambda seq: seq + ["Exit"])
)

print("Total sequences:", len(sequences))
print("\n5 sample sequences:")
for i, seq in enumerate(sequences.head(5), start=1):
    print(f"{i}. {seq[:12]}{' ...' if len(seq) > 12 else ''}")

KeyError: 'Column not found: state'

In [ ]:
## 5. Transition Matrix

transitions = []
for seq in sequences:
    transitions.extend(list(zip(seq[:-1], seq[1:])))

transitions_df = pd.DataFrame(transitions, columns=["from_state", "to_state"])
transition_counts = pd.crosstab(transitions_df["from_state"], transitions_df["to_state"])
transition_probs = transition_counts.div(transition_counts.sum(axis=1), axis=0)

print("Transition count table:")
print(transition_counts)
print("\nTransition probability matrix:")
print(transition_probs.round(4))

In [ ]:
# Heatmap visualization
plt.figure(figsize=(8, 5))
plt.imshow(transition_probs.values, aspect="auto", cmap="Blues")
plt.colorbar(label="Probability")
plt.xticks(range(len(transition_probs.columns)), transition_probs.columns, rotation=45)
plt.yticks(range(len(transition_probs.index)), transition_probs.index)
plt.title("Markov Transition Probability Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
## 6. Prediction

def predict_next_state(current_state: str, prob_matrix: pd.DataFrame = transition_probs) -> pd.Series:
    if current_state not in prob_matrix.index:
        raise ValueError(f"Unknown state: {current_state}. Choose from {list(prob_matrix.index)}")
    return prob_matrix.loc[current_state].sort_values(ascending=False)

print("Prediction from 'Browse':")
print(predict_next_state("Browse").round(4))

## 7. Insights

- Most users remain in `Browse` after a browse event.
- A smaller share moves from `Browse` to `Add_to_Cart`.
- `Purchase` commonly transitions to `Exit`, ending a journey.
- The Markov matrix can power lightweight next-step predictions.
- This can be extended to recommendations and conversion funnel monitoring.